# 验证内环闭合 (Closed-loop smoke test)

本 notebook 是 `README.md` 第三部分「验证内环闭合」的可执行版:跑一个 Bench2Drive 场景,然后查看 **metrics 评价分数** 与 **可视化视频**。看到分数 + 视频 = 内环闭合。

**前置条件**
- 在 `lead-eval` 容器内运行,内核选 `lead` conda 环境。
- `carla-server` 已启动(宿主机 `cd docker && ./start_carla.sh`)。
- 评测连 CARLA 用 `--host`(默认取环境变量 `CARLA_HOST`,双容器下即 `carla-server`)。

## 0. 环境与参数

In [ ]:
import json
import os
import shutil
import socket
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """从当前目录向上找含 pyproject.toml 的仓库根。"""
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("找不到仓库根(pyproject.toml)")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)  # 之后所有相对路径都相对仓库根
os.environ.setdefault("LEAD_PROJECT_ROOT", str(REPO_ROOT))

# ── 可调参数 ──
CHECKPOINT = "outputs/checkpoints/tfv6_resnet34"
ROUTES = "data/benchmark_routes/bench2drive/23687.xml"
CARLA_HOST = os.environ.get("CARLA_HOST", "carla-server")  # 双容器用 compose 服务名
CARLA_PORT = int(os.environ.get("CARLA_PORT", "2000"))
TIMEOUT = 300  # 单条 route 超时(秒)

ROUTE_ID = Path(ROUTES).stem.split("_")[0]
OUTPUT_DIR = REPO_ROOT / "outputs/local_evaluation" / ROUTE_ID

print("仓库根 :", REPO_ROOT)
print(f"CARLA  : {CARLA_HOST}:{CARLA_PORT}  | route_id = {ROUTE_ID}")
print("输出目录:", OUTPUT_DIR)

## 1. 确认 carla-server 可连

In [ ]:
import xml.etree.ElementTree as ET


def carla_reachable(host: str, port: int, timeout: float = 5.0) -> bool:
    """TCP 探测 CARLA RPC 端口是否可连。"""
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


def route_town(route_xml: Path) -> str:
    """从 route xml 读出场景所在 town(如 Town12)。"""
    el = ET.parse(route_xml).getroot().find(".//*[@town]")
    return el.get("town") if el is not None else ""


def carla_available_maps(host: str, port: int, timeout: float = 30.0) -> list:
    """连 CARLA 取 server 已加载地图清单(取末段名,如 Town12)。"""
    import carla

    client = carla.Client(host, port)
    client.set_timeout(timeout)
    return [m.rstrip("/").rsplit("/", 1)[-1] for m in client.get_available_maps()]


# ① 端口可连
ok = carla_reachable(CARLA_HOST, CARLA_PORT)
print("✅ carla 可连" if ok else "❌ carla 连不上")
assert ok, (
    f"连不上 {CARLA_HOST}:{CARLA_PORT}。请在宿主机执行:cd docker && ./start_carla.sh"
)

# ② server 真有本条 route 需要的地图。Bench2Drive 路线多在 Town12/13/15/11(附加地图),
#    基础 carla 镜像没有,缺图评测会报 Map 'TownXX' not found(见 docker/carla-server.Dockerfile)。
need_town = route_town(Path(ROUTES))
maps = carla_available_maps(CARLA_HOST, CARLA_PORT)
has_map = need_town in maps
print(f"本条 route 需要地图: {need_town}")
print(
    f"server 地图数: {len(maps)} | Town1x: {sorted(m for m in maps if m.startswith('Town1'))}"
)
print("✅ 地图就位" if has_map else f"❌ server 缺 {need_town}")
assert has_map, (
    f"server 没有地图 {need_town}:附加地图(Town11/12/13/15)未导入 carla-server。"
    f"请在宿主机重建带图镜像:cd docker && ./start_carla.sh"
)

## 2. 下载模型 checkpoint(已存在则跳过)

In [ ]:
ckpt = REPO_ROOT / CHECKPOINT
if (ckpt / "model_0030_0.pth").exists() and (ckpt / "config.json").exists():
    print("checkpoint 已就位,跳过下载。")
else:
    print("下载 checkpoint ...")
    subprocess.run(
        ["bash", "scripts/download_one_checkpoint.sh"], check=True, cwd=REPO_ROOT
    )
    print("下载完成。")

## 3. 跑一个场景(闭环评测)

连到 `carla-server` 跑完整条 route,通常几分钟。通过 `LEAD_CLOSED_LOOP_CONFIG` 开启视频录制
(demo / debug / grid)。底层就是 README 第三部分的 `python -m lead ... --bench2drive`,加上 `--host`。

In [ ]:
# 清掉旧结果,确保下面看到的分数/视频是本次产生的
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

env = os.environ.copy()
env["LEAD_PROJECT_ROOT"] = str(REPO_ROOT)
# 开启可视化视频录制(默认关闭,见 lead/inference/config_closed_loop.py 与 docs 第 3 节)
env["LEAD_CLOSED_LOOP_CONFIG"] = (
    "produce_demo_video=true produce_debug_video=true produce_grid_video=true"
)

cmd = [
    sys.executable,
    "-m",
    "lead",
    "--checkpoint",
    CHECKPOINT,
    "--routes",
    ROUTES,
    "--bench2drive",
    "--host",
    CARLA_HOST,
    "--port",
    str(CARLA_PORT),
    "--timeout",
    str(TIMEOUT),
]
print("运行:", " ".join(cmd), "\n")

# 实时回显子进程输出
proc = subprocess.Popen(
    cmd,
    cwd=REPO_ROOT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
ret = proc.wait()
print(f"\n评测进程退出码: {ret}")

## 4. metrics 评价分数

In [ ]:
# 标准 leaderboard 结果(驾驶分 / 路线完成度 / 违规惩罚)
ckpt_json = OUTPUT_DIR / "checkpoint_endpoint.json"
assert ckpt_json.exists(), f"没找到 {ckpt_json},评测可能未正常结束(看上一格日志)。"

data = json.loads(ckpt_json.read_text())
records = data.get("_checkpoint", {}).get("records", [])
print(f"完成 {len(records)} 条 route\n")
for rec in records:
    s = rec.get("scores", {})
    print(f"Route {rec.get('route_id')}  status={rec.get('status')}")
    print(f"  Driving Score   (score_composed): {s.get('score_composed')}")
    print(f"  Route Completion (score_route)  : {s.get('score_route')}")
    print(f"  Infraction Penalty(score_penalty): {s.get('score_penalty')}")
    nz = {k: v for k, v in rec.get("infractions", {}).items() if v}
    if nz:
        print("  违规:")
        for k, v in nz.items():
            print(f"    - {k}: {len(v) if isinstance(v, list) else v}")
    print()

glob = data.get("_checkpoint", {}).get("global_record", {})
if glob:
    print("== 全局汇总 scores ==")
    print(json.dumps(glob.get("scores", glob), indent=2, ensure_ascii=False))

# bench2drive 逐帧指标(可选)
metric_json = OUTPUT_DIR / "metric_info.json"
if metric_json.exists():
    m = json.loads(metric_json.read_text())
    print(f"\nmetric_info.json: {len(m)} 帧记录(bench2drive 逐帧指标)")
else:
    print("\n(无 metric_info.json)")

## 5. 可视化视频

closed-loop 视频用 OpenCV `mp4v` 编码,浏览器/Jupyter 常无法直接播放,这里用 `ffmpeg` 转成 H.264 再内嵌。

In [ ]:
from IPython.display import Video, display


def to_browser_mp4(src: Path) -> Path:
    """把 mp4v 视频转成浏览器可播的 H.264 (yuv420p);已转且较新则复用。"""
    dst = src.with_name(src.stem + "_h264.mp4")
    if dst.exists() and dst.stat().st_mtime >= src.stat().st_mtime:
        return dst
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-loglevel",
            "error",
            "-i",
            str(src),
            "-c:v",
            "libx264",
            "-pix_fmt",
            "yuv420p",
            "-movflags",
            "+faststart",
            str(dst),
        ],
        check=True,
    )
    return dst


candidates = [
    OUTPUT_DIR / f"{ROUTE_ID}_demo.mp4",
    OUTPUT_DIR / f"{ROUTE_ID}_debug.mp4",
    OUTPUT_DIR / f"{ROUTE_ID}_grid.mp4",
]
shown = False
for src in candidates:
    if src.exists() and src.stat().st_size > 0:
        print(f"▶ {src.name}")
        display(Video(str(to_browser_mp4(src)), embed=True, width=900))
        shown = True
if not shown:
    existing = (
        sorted(p.name for p in OUTPUT_DIR.glob("*")) if OUTPUT_DIR.exists() else []
    )
    print("没找到视频。确认 LEAD_CLOSED_LOOP_CONFIG 已开启录制且评测正常结束。")
    print("现有产物:", existing)